# LeetCode 838 — Push Dominoes bằng Python

Link đề: https://leetcode.com/problems/push-dominoes/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def pushDominoes(self, dominoes: str) -> str:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một chuỗi) nên so sánh bằng `==`. Notebook có oracle **mô phỏng từng giây** đúng theo luật của đề và đối chiếu mọi cách giải trên **mọi** chuỗi `L`, `R`, `.` độ dài đến 9, cùng các chuỗi ngẫu nhiên dài tới `10^5`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hàng domino đứng thẳng; ban đầu một số quân bị đẩy sang trái (`L`) hoặc phải (`R`) **đồng thời**; mỗi giây quân đang đổ đẩy quân kề bên theo hướng đổ; quân bị đẩy từ **cả hai phía** thì đứng yên (lực cân bằng). Tìm trạng thái cuối cùng.
2. Nắm các công cụ Python: chuỗi và list ký tự, `list("...")`/`"".join`, `collections.deque` (BFS nhiều nguồn), hai con trỏ, quét hai chiều (trái → phải và phải → trái) với mảng "lực".
3. Đi từ **mô phỏng từng giây (brute force)** → **BFS nhiều nguồn có đánh dấu thời gian** → **hai lượt quét tính lực** → **chia đoạn giữa hai quân bị đẩy liền kề** (hai con trỏ).
4. Hiểu mấu chốt: tác động của mỗi quân bị đẩy **không đi xa hơn quân bị đẩy kế tiếp**; vì vậy bài tách thành các **đoạn độc lập** giữa hai ký tự `L`/`R` liền kề: `R…L` (gặp nhau: chia đôi, ô giữa đứng yên nếu số ô lẻ), `L…L` và `R…R` (cùng một hướng: đổ hết), `L…R` (hai bên đẩy ra xa nhau: đứng yên).
5. Tránh các lỗi kinh điển: cập nhật **tại chỗ** (không đồng thời), quên ô giữa đứng yên khi số ô lẻ, quên biên (`L` ở đầu kéo mọi `.` phía trước đổ trái, `R` ở cuối đẩy mọi `.` phía sau), để lực `R` lan qua một `L`, và BFS không phát hiện va chạm cùng thời điểm.

## 2. Đề bài & ví dụ

Có `n` quân domino xếp thành hàng, mỗi quân đặt **thẳng đứng**. Ban đầu ta **đồng thời** đẩy một số quân sang trái hoặc sang phải.

Sau mỗi giây, mỗi quân **đang đổ sang trái** đẩy quân kề bên **trái**; tương tự, quân đang đổ sang phải đẩy quân kề bên **phải**.

Khi một quân đang đứng thẳng bị các quân đổ **từ cả hai phía** đẩy vào cùng lúc, nó **đứng yên** vì lực cân bằng.

Trong bài này coi rằng một quân đang đổ **không truyền thêm lực** cho quân đang đổ hoặc đã đổ.

Cho chuỗi `dominoes` mô tả trạng thái đầu:

- `'L'`: quân thứ `i` bị đẩy sang trái;
- `'R'`: quân thứ `i` bị đẩy sang phải;
- `'.'`: quân thứ `i` không bị đẩy.

Trả về chuỗi mô tả trạng thái cuối.

**Ví dụ 1:** `"RR.L"` → `"RR.L"`. Quân `R` đầu không truyền thêm lực cho quân thứ hai (đã đổ); quân giữa `.` bị `R` đẩy từ trái và `L` đẩy từ phải cùng lúc nên đứng yên.

**Ví dụ 2:** `".L.R...LR..L.."` → `"LL.RR.LLRRLL.."`.

**Ràng buộc:** `n == dominoes.length`, `1 <= n <= 10^5`, `dominoes[i]` là `'L'`, `'R'` hoặc `'.'`.

In [1]:
import random
import time
from collections import deque
from itertools import product
from typing import List

ex1 = ("RR.L", "RR.L")
ex2 = (".L.R...LR..L..", "LL.RR.LLRRLL..")
for s, want in (ex1, ex2):
    print(f"{s}  ->  {want}")

RR.L  ->  RR.L
.L.R...LR..L..  ->  LL.RR.LLRRLL..


## 3. Kiến thức nền

### 3.1 Chuỗi là bất biến, ta dùng list ký tự

Chuỗi Python **không sửa tại chỗ được**: `s[2] = 'L'` gây lỗi. Muốn sửa nhiều vị trí, đổi thành list: `a = list(s)`, sửa `a[i]`, rồi `"".join(a)` ghép lại.

In [2]:
s = "R..L"
a = list(s)
a[1] = "R"
a[2] = "L"
print(s, "->", "".join(a))
try:
    s[1] = "R"
except TypeError as e:
    print("TypeError:", e)

R..L -> RRLL
TypeError: 'str' object does not support item assignment


### 3.2 Cập nhật **đồng thời** (simultaneous update)

Nhiều bài mô phỏng đòi mọi phần tử cập nhật **cùng lúc** dựa trên trạng thái **cũ**. Nếu cập nhật tại chỗ rồi đọc luôn giá trị mới của hàng xóm, hiệu ứng sẽ lan **nhanh hơn** thực tế trong cùng một bước. Phải tạo `new` từ `old` rồi mới gán.

Ví dụ `"R..."` mỗi giây thực tế chỉ có **một** quân đổ thêm: `"RR.." → "RRR." → "RRRR"`. Nếu cập nhật tại chỗ từ trái sang phải thì sau **một** giây ra ngay `"RRRR"`.

In [3]:
def step_simultaneous(s):
    old = list(s)
    new = old[:]
    for i, c in enumerate(old):
        if c == ".":
            left_R = i > 0 and old[i - 1] == "R"
            right_L = i + 1 < len(old) and old[i + 1] == "L"
            if left_R and not right_L:
                new[i] = "R"
            elif right_L and not left_R:
                new[i] = "L"
    return "".join(new)


def step_inplace(s):                                  # SAI: đọc giá trị vừa cập nhật
    a = list(s)
    for i, c in enumerate(a):
        if c == ".":
            left_R = i > 0 and a[i - 1] == "R"
            right_L = i + 1 < len(a) and a[i + 1] == "L"
            if left_R and not right_L:
                a[i] = "R"
            elif right_L and not left_R:
                a[i] = "L"
    return "".join(a)


print("đồng thời :", step_simultaneous("R..."), "(chỉ 1 quân đổ thêm)")
print("tại chỗ   :", step_inplace("R..."), "(lan hết ngay trong 1 giây: sai)")

đồng thời : RR.. (chỉ 1 quân đổ thêm)
tại chỗ   : RRRR (lan hết ngay trong 1 giây: sai)


### 3.3 BFS nhiều nguồn

**BFS** (duyệt theo chiều rộng) xử lý các nút theo **lớp khoảng cách**. **BFS nhiều nguồn** đặt nhiều nút xuất phát vào hàng đợi cùng lúc (ở đây: mọi quân `L`/`R` ban đầu, thời điểm `0`). Khi duyệt, ta ghi lại **thời điểm** mỗi quân đổ: các quân cùng lớp đổ cùng giây. `collections.deque` cho `popleft()` `O(1)`.

### 3.4 Quét hai chiều và mảng "lực"

Gán cho mỗi quân một **lực** là số nguyên: lực đẩy phải dương, lực đẩy trái âm; lực giảm dần theo khoảng cách. Quét trái → phải tính lực `R` truyền tới từng ô; quét phải → trái tính lực `L`; tổng hai lực quyết định hướng: `> 0` → `R`, `< 0` → `L`, `= 0` → `.`.

### 3.5 Hai con trỏ trên các điểm "có lực"

Duyệt chuỗi, giữ con trỏ tới ký tự `L`/`R` liền trước và liền sau; xử lý đoạn `.` ở giữa theo cặp (hướng trước, hướng sau). Mỗi ô chỉ ảnh hưởng bởi hai quân bị đẩy gần nhất hai bên nên đoạn nào độc lập với đoạn khác.

In [4]:
s = ".L.R...LR..L.."
marks = [(i, c) for i, c in enumerate(s) if c != "."]
print("các quân bị đẩy:", marks)
print("các cặp liền kề :", list(zip(marks, marks[1:])))

các quân bị đẩy: [(1, 'L'), (3, 'R'), (7, 'L'), (8, 'R'), (11, 'L')]
các cặp liền kề : [((1, 'L'), (3, 'R')), ((3, 'R'), (7, 'L')), ((7, 'L'), (8, 'R')), ((8, 'R'), (11, 'L'))]


## 4. Ý tưởng nền tảng / trực giác

Xét một đoạn `.` nằm giữa hai quân bị đẩy liền kề (hoặc giữa quân bị đẩy và đầu/cuối hàng). Ta chỉ cần biết hai ký tự hai bên:

| bên trái | bên phải | điều xảy ra với đoạn `.` ở giữa |
|---|---|---|
| `R` | `L` | hai lực gặp nhau: nửa đầu đổ phải, nửa sau đổ trái; ô giữa đứng yên nếu số ô lẻ |
| `R` | `R` | `R` đẩy hết các quân `.` sang phải (cuối cùng cả đoạn là `R`) |
| `L` | `L` | quân `L` bên phải kéo hết đoạn sang trái |
| `L` | `R` | hai quân đẩy **ra xa** nhau: đoạn `.` ở giữa đứng yên |
| (đầu hàng) | `L` | `L` kéo hết các `.` phía trước sang trái |
| `R` | (cuối hàng) | `R` đẩy hết các `.` phía sau sang phải |
| (đầu hàng) | `R` hoặc `L` … | `.` đầu hàng bên trái `R`: đứng yên (không ai đẩy tới) |

Tính tay `".L.R...LR..L.."`:

- Trước `L` đầu tiên (đầu hàng, 1 dấu `.`): thành `L`.
- `L` rồi `.` rồi `R` (1 dấu chấm giữa `L` và `R`): đứng yên `.`.
- `R...L` (3 dấu chấm giữa `R` và `L`): `RR.LL`.
- `LR` kề nhau: giữ nguyên.
- `R..L` (2 dấu chấm): `RRLL`.
- Sau `L` cuối (2 dấu chấm, tới cuối hàng): đứng yên (không có `R` phía sau).

Ghép lại: `LL` `.` `RR.LL` `LR` `RRLL` `..` = `"LL.RR.LLRRLL.."` ✓.

In [5]:
def show_segments(s):
    marks = [(-1, "|")] + [(i, c) for i, c in enumerate(s) if c != "."] + [(len(s), "|")]
    for (i, a), (j, b) in zip(marks, marks[1:]):
        gap = j - i - 1
        print(f"  giữa {a!r}@{i} và {b!r}@{j}: {gap} ô '.'  ", end="")
        if a in "R" and b == "L":
            print(f"-> gặp nhau: {gap // 2} ô mỗi bên đổ, {gap % 2} ô giữa đứng yên")
        elif a == b and a != "|":
            print(f"-> cả đoạn đổ về phía {a}")
        elif a == "|" and b == "L":
            print("-> đổ trái (kéo bởi L ở đầu đoạn phải)")
        elif a == "R" and b == "|":
            print("-> đổ phải")
        else:
            print("-> đứng yên")


show_segments(".L.R...LR..L..")

  giữa '|'@-1 và 'L'@1: 1 ô '.'  -> đổ trái (kéo bởi L ở đầu đoạn phải)
  giữa 'L'@1 và 'R'@3: 1 ô '.'  -> đứng yên
  giữa 'R'@3 và 'L'@7: 3 ô '.'  -> gặp nhau: 1 ô mỗi bên đổ, 1 ô giữa đứng yên
  giữa 'L'@7 và 'R'@8: 0 ô '.'  -> đứng yên
  giữa 'R'@8 và 'L'@11: 2 ô '.'  -> gặp nhau: 1 ô mỗi bên đổ, 0 ô giữa đứng yên
  giữa 'L'@11 và '|'@14: 2 ô '.'  -> đứng yên


## 5. Cách 1 — Brute Force: mô phỏng từng giây

**Ý tưởng:** làm đúng như đề. Mỗi giây tạo trạng thái mới từ trạng thái cũ **đồng thời**: quân `.` đổi thành `R` nếu chỉ có hàng xóm trái là `R` (và hàng xóm phải không phải `L`); đổi thành `L` nếu chỉ hàng xóm phải là `L`; còn lại giữ nguyên. Lặp tới khi không còn thay đổi.

In [6]:
def push_brute(dominoes):
    s = dominoes
    seconds = 0
    while True:
        t = step_simultaneous(s)
        if t == s:
            break
        s = t
        seconds += 1
    push_brute.seconds = seconds
    return s


for s, want in (ex1, ex2):
    got = push_brute(s)
    print(f"{s} -> {got} ({push_brute.seconds} giây có thay đổi)")
    assert got == want

RR.L -> RR.L (0 giây có thay đổi)
.L.R...LR..L.. -> LL.RR.LLRRLL.. (1 giây có thay đổi)


**Độ phức tạp:** mỗi giây `O(n)`, số giây tối đa `O(n)` (ví dụ `"R" + "." * (n − 1)`: mỗi giây đổ thêm một quân) → `O(n²)`; `n = 10^5` là `10^10` phép: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** ta quét lại cả những quân **đã ổn định** ở mỗi giây. Thực ra chỉ những quân ở **ranh giới** đang đổ (đầu sóng) mới thay đổi.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Đoạn độc lập.** Lực từ một quân bị đẩy tới ô `i` bị **chặn** bởi quân bị đẩy gần hơn (một quân `L`/`R` đã tự đổ, không truyền lực thêm). Do đó kết quả của mỗi ô chỉ phụ thuộc hai quân bị đẩy **gần nhất** ở hai phía (một bên trái, một bên phải). Ta xử lý từng đoạn `.` giữa hai quân bị đẩy liền kề như bảng ở mục 4.
2. **Có thể tính ngay không cần mô phỏng thời gian.** Với đoạn `R…L` có `g` ô: ô thứ `t` (từ trái, `t = 1..g`) đổ phải nếu `t ≤ g/2`, đổ trái nếu `t > g − g/2`, và đứng yên nếu là ô giữa khi `g` lẻ.

Từ đây có ba cách giải: BFS với thời gian (mô phỏng theo sóng), hai lượt quét "lực", hoặc chia đoạn bằng hai con trỏ.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** BFS nhiều nguồn có thời gian, **Cách 3** hai lượt quét tính lực, **Cách 4** chia đoạn hai con trỏ.

### Cách 2 — BFS nhiều nguồn có đánh dấu thời gian

**Giải thích từng đoạn:**

1. Đưa mọi quân `L`/`R` vào hàng đợi với thời điểm `0`, ghi `force[i]` = danh sách hướng tác động lên quân `i`.
2. Lấy quân `i` ra. Nếu nó có **hai** lực (đã cân bằng) thì **không lan tiếp**. Ngược lại hướng `f = force[i][0]`; quân kề `j = i − 1` (nếu `L`) hoặc `i + 1` (nếu `R`).
3. Nếu `j` hợp lệ: nếu chưa có thời điểm → đặt thời điểm `time[i] + 1`, ghi hướng, đặt kết quả `f`, đưa vào hàng đợi. Nếu `j` đã có thời điểm **đúng bằng** `time[i] + 1` → lực thứ hai (ngược chiều) tới cùng lúc: ghi hướng thứ hai và đặt kết quả `'.'` (cân bằng). Nếu thời điểm sớm hơn thì `j` đã đổ từ trước: bỏ qua.

In [7]:
def push_bfs(dominoes):
    n = len(dominoes)
    time_at = [-1] * n
    force = [[] for _ in range(n)]
    ans = list(dominoes)
    q = deque()
    for i, c in enumerate(dominoes):
        if c != ".":
            q.append(i)
            time_at[i] = 0
            force[i].append(c)
    while q:
        i = q.popleft()
        if len(force[i]) == 2:                       # lực cân bằng: không lan tiếp
            continue
        f = force[i][0]
        j = i - 1 if f == "L" else i + 1
        if 0 <= j < n:
            t = time_at[i]
            if time_at[j] == -1:                     # lần đầu có lực tới j
                time_at[j] = t + 1
                force[j].append(f)
                ans[j] = f
                q.append(j)
            elif time_at[j] == t + 1:                # lực ngược chiều tới CÙNG giây
                force[j].append(f)
                ans[j] = "."
    return "".join(ans)


for s, want in (ex1, ex2):
    assert push_bfs(s) == want
print(push_bfs(ex2[0]))

LL.RR.LLRRLL..


In [8]:
# Code nộp LeetCode cho Cách 2
from collections import deque


class SolutionBFS:
    def pushDominoes(self, dominoes: str) -> str:
        n = len(dominoes)
        time_at = [-1] * n
        force = [[] for _ in range(n)]
        ans = list(dominoes)
        q = deque()
        for i, c in enumerate(dominoes):
            if c != ".":
                q.append(i)
                time_at[i] = 0
                force[i].append(c)
        while q:
            i = q.popleft()
            if len(force[i]) == 2:
                continue
            f = force[i][0]
            j = i - 1 if f == "L" else i + 1
            if 0 <= j < n:
                t = time_at[i]
                if time_at[j] == -1:
                    time_at[j] = t + 1
                    force[j].append(f)
                    ans[j] = f
                    q.append(j)
                elif time_at[j] == t + 1:
                    force[j].append(f)
                    ans[j] = "."
        return "".join(ans)

**Độ phức tạp:** mỗi quân vào hàng đợi tối đa một lần: `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 3 — Hai lượt quét tính lực

**Giải thích từng đoạn:**

1. Lượt trái → phải: `fr[i]` = lực `R` tại `i`: `n` nếu `s[i] = 'R'`; `0` nếu `s[i] = 'L'` (chặn); ngược lại `max(fr[i−1] − 1, 0)` (lực giảm `1` mỗi ô).
2. Lượt phải → trái: `fl[i]` = lực `L` (lấy giá trị dương): `n` nếu `'L'`; `0` nếu `'R'`; ngược lại `max(fl[i+1] − 1, 0)`.
3. Quân `i`: `fr[i] − fl[i] > 0` → `R`; `< 0` → `L`; `= 0` → `.`.

Lực giảm theo khoảng cách nên quân bị đẩy **gần hơn** thắng; cân bằng khi cách đều hai nguồn đối nghịch.

In [9]:
def push_force(dominoes):
    n = len(dominoes)
    fr = [0] * n
    fl = [0] * n
    f = 0
    for i, c in enumerate(dominoes):               # lực R từ trái sang
        if c == "R":
            f = n
        elif c == "L":
            f = 0
        else:
            f = max(f - 1, 0)
        fr[i] = f
    f = 0
    for i in range(n - 1, -1, -1):                 # lực L từ phải sang
        c = dominoes[i]
        if c == "L":
            f = n
        elif c == "R":
            f = 0
        else:
            f = max(f - 1, 0)
        fl[i] = f
    return "".join("R" if fr[i] > fl[i] else "L" if fr[i] < fl[i] else "." for i in range(n))


for s, want in (ex1, ex2):
    assert push_force(s) == want
print(push_force(ex2[0]))

LL.RR.LLRRLL..


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionForce:
    def pushDominoes(self, dominoes: str) -> str:
        n = len(dominoes)
        right = [0] * n
        left = [0] * n
        f = 0
        for i, c in enumerate(dominoes):
            if c == "R":
                f = n
            elif c == "L":
                f = 0
            else:
                f = max(f - 1, 0)
            right[i] = f
        f = 0
        for i in range(n - 1, -1, -1):
            c = dominoes[i]
            if c == "L":
                f = n
            elif c == "R":
                f = 0
            else:
                f = max(f - 1, 0)
            left[i] = f
        return "".join("R" if right[i] > left[i] else "L" if right[i] < left[i] else "." for i in range(n))

**Độ phức tạp:** hai lượt quét `O(n)`; bộ nhớ `O(n)`.

### Cách 4 — Chia đoạn bằng hai con trỏ (đáp án chính)

**Ý tưởng:** thêm **lính canh**: `'L'` ở đầu và `'R'` ở cuối chuỗi. Tại sao? Lính `L` ở đầu: nếu ký tự bị đẩy đầu tiên thật là `L` thì đoạn `.` phía trước (giữa lính `L` và nó) rơi vào ca `L…L` → đổ trái ✓; nếu là `R` thì ca `L…R` → đứng yên ✓ (đúng: không có gì đẩy `.` ở đầu). Tương tự lính `R` ở cuối: nếu ký tự cuối là `R` thì `R…R` đổ phải ✓; nếu là `L` thì `L…R` đứng yên ✓.

Duyệt từ trái với con trỏ `i` tại ký tự bị đẩy hiện tại, tìm `j` là ký tự bị đẩy kế tiếp (bỏ qua `.`); xử lý cặp `(s[i], s[j])`:

- `L…L`: điền `L` cho mọi ô giữa.
- `R…R`: điền `R`.
- `L…R`: giữ `.`.
- `R…L`: điền `R` cho nửa đầu `(j − i − 1) // 2` ô, `L` cho nửa sau cùng số ô; ô giữa (nếu số ô lẻ) giữ `.`.

In [11]:
def push_segments(dominoes):
    s = "L" + dominoes + "R"                        # hai lính canh
    ans = []
    i = 0
    for j in range(1, len(s)):
        if s[j] == ".":
            continue
        gap = j - i - 1
        if i:                                        # không thêm lính canh trái vào kết quả
            ans.append(s[i])
        if s[i] == s[j]:
            ans.append(s[i] * gap)                   # L..L hoặc R..R
        elif s[i] == "L" and s[j] == "R":
            ans.append("." * gap)                    # đẩy ra xa nhau
        else:                                        # R..L: gặp nhau
            ans.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))
        i = j
    return "".join(ans)


for s, want in (ex1, ex2):
    assert push_segments(s) == want
print(push_segments(ex2[0]), push_segments("."), push_segments("L"), push_segments("R"))
assert push_segments(".") == "." and push_segments("L") == "L" and push_segments("R") == "R"

LL.RR.LLRRLL.. . L R


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionSegments:
    def pushDominoes(self, dominoes: str) -> str:
        s = "L" + dominoes + "R"
        ans = []
        i = 0
        for j in range(1, len(s)):
            if s[j] == ".":
                continue
            gap = j - i - 1
            if i:
                ans.append(s[i])
            if s[i] == s[j]:
                ans.append(s[i] * gap)
            elif s[i] == "L" and s[j] == "R":
                ans.append("." * gap)
            else:
                ans.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))
            i = j
        return "".join(ans)

**Độ phức tạp:** mỗi ký tự được duyệt một lần: `O(n)` thời gian; `O(n)` bộ nhớ cho kết quả (chuỗi ghép từ các đoạn).

## 8. TRACE từng bước

**Trace Cách 4 (chia đoạn)** với `".L.R...LR..L.."`. Thêm lính canh: `s = "L" + ".L.R...LR..L.." + "R"`. Các ký tự bị đẩy (trong `s`, kể cả hai lính canh) ở các chỉ số: `0 (L lính)`, `2 (L)`, `4 (R)`, `8 (L)`, `9 (R)`, `12 (L)`, `15 (R lính)`. Các cặp liền kề và đoạn `.` giữa chúng:

| cặp `(s[i], s[j])` | số ô `.` giữa | kết quả đoạn |
|---|---|---|
| `L@0, L@2` | 1 | `L` (cả đoạn đổ trái) |
| `L@2, R@4` | 1 | `.` (đẩy ra xa nhau) |
| `R@4, L@8` | 3 | `RR.LL` — gồm `R`, `R`, `.`, `L`, rồi `L@8` |
| `L@8, R@9` | 0 | (rỗng) |
| `R@9, L@12` | 2 | `RRLL` |
| `L@12, R@15` | 2 | `..` (đứng yên) |

Hàm trace in từng đoạn (bỏ phần thêm lính canh khỏi kết quả):

In [13]:
def trace_segments(dominoes):
    s = "L" + dominoes + "R"
    print("chuỗi có lính canh:", s)
    ans = []
    i = 0
    for j in range(1, len(s)):
        if s[j] == ".":
            continue
        gap = j - i - 1
        if s[i] == s[j]:
            seg, why = s[i] * gap, f"{s[i]}..{s[j]}: cả đoạn đổ {s[i]}"
        elif s[i] == "L" and s[j] == "R":
            seg, why = "." * gap, "L..R: đẩy ra xa nhau, đứng yên"
        else:
            seg, why = "R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2), f"R..L: {gap // 2} ô đổ R, {gap % 2} ô giữa, {gap // 2} ô đổ L"
        print(f"  cặp ({s[i]!r}@{i}, {s[j]!r}@{j}), {gap} ô '.': {why} -> đoạn {seg!r}")
        if i:
            ans.append(s[i])
        ans.append(seg)
        i = j
    res = "".join(ans)
    print("kết quả:", res)
    return res


assert trace_segments(ex2[0]) == ex2[1]

chuỗi có lính canh: L.L.R...LR..L..R
  cặp ('L'@0, 'L'@2), 1 ô '.': L..L: cả đoạn đổ L -> đoạn 'L'
  cặp ('L'@2, 'R'@4), 1 ô '.': L..R: đẩy ra xa nhau, đứng yên -> đoạn '.'
  cặp ('R'@4, 'L'@8), 3 ô '.': R..L: 1 ô đổ R, 1 ô giữa, 1 ô đổ L -> đoạn 'R.L'
  cặp ('L'@8, 'R'@9), 0 ô '.': L..R: đẩy ra xa nhau, đứng yên -> đoạn ''
  cặp ('R'@9, 'L'@12), 2 ô '.': R..L: 1 ô đổ R, 0 ô giữa, 1 ô đổ L -> đoạn 'RL'
  cặp ('L'@12, 'R'@15), 2 ô '.': L..R: đẩy ra xa nhau, đứng yên -> đoạn '..'
kết quả: LL.RR.LLRRLL..


**Trace Cách 3 (lực)** cho `"R..L"` (`n = 4`): lượt trái → phải: `R` đặt `f = 4`; ô `1`: `3`; ô `2`: `2`; ô `3` là `L` đặt `f = 0`. `fr = [4, 3, 2, 0]`. Lượt phải → trái: `L@3` đặt `f = 4`; ô `2`: `3`; ô `1`: `2`; `R@0` đặt `0`. `fl = [0, 2, 3, 4]`. So sánh: ô0 `4 > 0` → `R`; ô1 `3 > 2` → `R`; ô2 `2 < 3` → `L`; ô3 `0 < 4` → `L` → `"RRLL"` ✓. Hàm trace in hai mảng lực:

In [14]:
def trace_force(dominoes):
    n = len(dominoes)
    fr, fl = [0] * n, [0] * n
    f = 0
    for i, c in enumerate(dominoes):
        f = n if c == "R" else 0 if c == "L" else max(f - 1, 0)
        fr[i] = f
    f = 0
    for i in range(n - 1, -1, -1):
        c = dominoes[i]
        f = n if c == "L" else 0 if c == "R" else max(f - 1, 0)
        fl[i] = f
    res = "".join("R" if fr[i] > fl[i] else "L" if fr[i] < fl[i] else "." for i in range(n))
    print("domino :", " ".join(f"{c:>2}" for c in dominoes))
    print("lực R  :", " ".join(f"{v:>2}" for v in fr))
    print("lực L  :", " ".join(f"{v:>2}" for v in fl))
    print("kết quả:", " ".join(f"{c:>2}" for c in res))
    return res


assert trace_force("R..L") == "RRLL"
assert trace_force("R...L") == "RR.LL"
assert trace_force("RR.L") == "RR.L"

domino :  R  .  .  L
lực R  :  4  3  2  0
lực L  :  0  2  3  4
kết quả:  R  R  L  L
domino :  R  .  .  .  L
lực R  :  5  4  3  2  0
lực L  :  0  2  3  4  5
kết quả:  R  R  .  L  L
domino :  R  R  .  L
lực R  :  4  4  3  0
lực L  :  0  0  3  4
kết quả:  R  R  .  L


**Trace Cách 2 (BFS)** cho `"R...L"`: thời điểm `0`: `R@0`, `L@4`. Giây `1`: `R` đổ ô `1`, `L` đổ ô `3` (đều có thời điểm `1`). Giây `2`: `R@1` thử đổ ô `2`: ô `2` chưa có thời điểm → đặt `time = 2`, hướng `R`; `L@3` thử đổ ô `2`: ô `2` đã có `time = 2 = 1 + 1` → lực thứ hai → ô `2` `'.'` (cân bằng). Kết quả `"RR.LL"`.

In [15]:
def trace_bfs(dominoes):
    n = len(dominoes)
    time_at = [-1] * n
    force = [[] for _ in range(n)]
    ans = list(dominoes)
    q = deque()
    for i, c in enumerate(dominoes):
        if c != ".":
            q.append(i); time_at[i] = 0; force[i].append(c)
    print("khởi tạo hàng đợi:", list(q))
    while q:
        i = q.popleft()
        if len(force[i]) == 2:
            print(f"lấy {i}: có 2 lực (cân bằng) -> không lan tiếp")
            continue
        f = force[i][0]
        j = i - 1 if f == "L" else i + 1
        msg = f"lấy {i} (thời điểm {time_at[i]}, hướng {f}) -> thử quân {j}: "
        if not 0 <= j < n:
            msg += "ngoài hàng"
        elif time_at[j] == -1:
            time_at[j] = time_at[i] + 1; force[j].append(f); ans[j] = f; q.append(j)
            msg += f"chưa có lực -> đặt thời điểm {time_at[j]}, đổ {f}"
        elif time_at[j] == time_at[i] + 1:
            force[j].append(f); ans[j] = "."
            msg += f"đã có lực cùng giây {time_at[j]} -> cân bằng '.'"
        else:
            msg += "đã đổ từ trước -> bỏ qua"
        print(msg)
    print("kết quả:", "".join(ans))
    return "".join(ans)


assert trace_bfs("R...L") == "RR.LL"

khởi tạo hàng đợi: [0, 4]
lấy 0 (thời điểm 0, hướng R) -> thử quân 1: chưa có lực -> đặt thời điểm 1, đổ R
lấy 4 (thời điểm 0, hướng L) -> thử quân 3: chưa có lực -> đặt thời điểm 1, đổ L
lấy 1 (thời điểm 1, hướng R) -> thử quân 2: chưa có lực -> đặt thời điểm 2, đổ R
lấy 3 (thời điểm 1, hướng L) -> thử quân 2: đã có lực cùng giây 2 -> cân bằng '.'
lấy 2: có 2 lực (cân bằng) -> không lan tiếp
kết quả: RR.LL


## 9. Code nộp LeetCode

Chọn **chia đoạn bằng hai con trỏ** (gọn, `O(n)`). Giữ nguyên `class Solution`, `pushDominoes(self, dominoes)`:

In [16]:
class Solution:
    def pushDominoes(self, dominoes: str) -> str:
        s = "L" + dominoes + "R"            # lính canh: L ở đầu, R ở cuối
        ans = []
        i = 0                               # i: quân bị đẩy liền trước (tính cả lính canh)
        for j in range(1, len(s)):
            if s[j] == ".":
                continue
            gap = j - i - 1                 # số ô '.' giữa i và j
            if i:
                ans.append(s[i])            # quân bị đẩy thật (không thêm lính canh trái)
            if s[i] == s[j]:
                ans.append(s[i] * gap)                                          # L..L hoặc R..R: đổ cùng hướng
            elif s[i] == "L" and s[j] == "R":
                ans.append("." * gap)                                           # đẩy ra xa nhau: đứng yên
            else:
                ans.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))   # R..L: gặp nhau
            i = j
        return "".join(ans)


print(Solution().pushDominoes("RR.L"))
print(Solution().pushDominoes(".L.R...LR..L.."))
assert Solution().pushDominoes("RR.L") == "RR.L"
assert Solution().pushDominoes(".L.R...LR..L..") == "LL.RR.LLRRLL.."

RR.L
LL.RR.LLRRLL..


## 10. Vì sao đúng

**(a) Tính độc lập theo đoạn.** Gọi `p_1 < p_2 < … < p_t` là các vị trí có `L`/`R` ban đầu (các quân bị đẩy). Xét một ô `.` ở vị trí `x`, với `p_a < x < p_{a+1}`. Lực từ `p_b` với `b < a` muốn tới `x` (đổ phải) phải đi qua `p_a`: nếu `s[p_a] = 'R'` thì `p_a` đã đổ phải và truyền lực cho `x` (nhờ `p_a` cũng đổ phải, `p_b` không cần thêm), còn nếu `s[p_a] = 'L'` thì `p_a` đổ trái nên **không** cho lực phải đi qua (một quân đổ không truyền lực ngược hướng). Do đó chỉ `p_a` (bên trái) và `p_{a+1}` (bên phải) ảnh hưởng tới `x`; các quân bị đẩy xa hơn bị chặn.

**(b) Phân tích 4 trường hợp (đoạn có `g` ô `.`, đánh số `1..g` từ trái).**

- `s[p_a] = 'R'`, `s[p_{a+1}] = 'L'`: `R` tới ô `t` ở giây `t`, `L` tới ô `t` ở giây `g + 1 − t`. Quân `.` đổ theo lực tới **trước**: `t < g + 1 − t` ⇔ `t < (g + 1)/2` → `R`; `t > (g + 1)/2` → `L`; `t = (g + 1)/2` (chỉ khi `g` lẻ) → cùng giây → cân bằng, đứng yên. Do đó nửa đầu `g // 2` ô là `R`, nửa sau `g // 2` ô là `L`, ô giữa `.` khi `g` lẻ.
- `R…R` : `R` bên trái tới mọi ô (không có lực ngược nào, vì `R` bên phải đẩy sang phải, rời xa) → cả đoạn `R`.
- `L…L`: tương tự → cả đoạn `L`.
- `L…R` (trái là `L` đổ trái, phải là `R` đổ phải): không lực nào hướng vào đoạn → đứng yên.

**(c) Lính canh.** Thêm `L` ở đầu và `R` ở cuối biến mọi đoạn biên thành 4 trường hợp trên mà **không đổi đáp án**: đoạn đầu giữa lính `L` và quân bị đẩy đầu tiên: nếu nó là `L` → `L…L` cả đoạn đổ trái (đúng, quân `L` kéo các quân `.` phía trước), nếu là `R` → `L…R` đứng yên (đúng). Đoạn cuối tương tự.

**(d) Hai cách còn lại.** *Lực:* `fr[i]` là `n − (khoảng cách tới R gần nhất bên trái, không bị L chặn)` hoặc `0`; `fl[i]` tương tự; so sánh lực cho quân gần hơn thắng, hòa → đứng yên — cùng quy tắc "tới trước thắng, cùng lúc hòa" ở (b). *BFS:* các quân đổ theo lớp thời gian; va chạm cùng thời điểm ghi nhận cân bằng.

Ta kiểm chứng bằng máy: mọi cách khớp **mô phỏng từng giây** trên **mọi** chuỗi `L/R/.` độ dài ≤ 9 và nhiều chuỗi dài hơn (mục 12D–E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng giây | `O(n²)` | `O(n)` | oracle; `10^10` ở `n = 10^5` |
| 2. BFS nhiều nguồn | `O(n)` | `O(n)` | theo dõi thời gian đổ |
| 3. Hai lượt quét lực | `O(n)` | `O(n)` | rất ngắn, dễ nhớ |
| 4. Chia đoạn hai con trỏ | `O(n)` | `O(n)` | **đáp án chính** |

Đo thực tế (`n = 10^5`, kèm ca xấu nhất cho mô phỏng):

In [17]:
def random_dominoes(rng, n, weights=(1, 1, 8)):
    return "".join(rng.choices("LR.", weights=weights, k=n))


def bench(fn, s, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(s)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(838)
big = random_dominoes(rng, 10 ** 5)
ref = push_segments(big)
for name, fn in [("bfs", push_bfs), ("force", push_force), ("segments", push_segments)]:
    assert fn(big) == ref
    print(f"n = 10^5 {name:9s} {bench(fn, big) * 1000:8.2f} ms")
worst = "R" + "." * 4000
t0 = time.perf_counter()
res = push_brute(worst)
print(f"brute ca xấu nhất n = 4001 ('R' + '.'*4000): {(time.perf_counter() - t0) * 1000:.0f} ms, {push_brute.seconds} giây mô phỏng -> n = 10^5 khoảng {(time.perf_counter() - t0) * 625:.0f} s")
assert res == "R" * 4001 and push_segments(worst) == res

n = 10^5 bfs          40.50 ms
n = 10^5 force        29.35 ms
n = 10^5 segments      6.32 ms


brute ca xấu nhất n = 4001 ('R' + '.'*4000): 1787 ms, 4000 giây mô phỏng -> n = 10^5 khoảng 1117 s


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn mọi chuỗi** độ dài ≤ 9, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn lớn nhất. Oracle: `push_brute` (mô phỏng từng giây đồng thời).

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "brute": push_brute,
    "bfs": push_bfs,
    "force": push_force,
    "segments": push_segments,
    "Solution": lambda s: Solution().pushDominoes(s),
    "SolutionBFS": lambda s: SolutionBFS().pushDominoes(s),
    "SolutionForce": lambda s: SolutionForce().pushDominoes(s),
    "SolutionSegments": lambda s: SolutionSegments().pushDominoes(s),
}
FAST = [k for k in IMPLS if k != "brute"]


def check(s, want=None, names=None):
    if want is None:
        want = push_brute(s)
    for name in (names or IMPLS):
        got = IMPLS[name](s)
        assert got == want, (name, s, got, want)


for s, want in (ex1, ex2):
    check(s, want)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 8 cài đặt đúng


### B. Edge case

- Một quân: `"."`, `"L"`, `"R"`.
- Toàn `.` (không ai đẩy) → giữ nguyên.
- `"R."` → `"RR"`; `".L"` → `"LL"`; `"L."` → `"L."`; `".R"` → `".R"`.
- `"R.L"` (một ô giữa) → ô giữa đứng yên; `"R..L"` → `"RRLL"`; `"R...L"` → `"RR.LL"`.
- `"L.R"` → giữ nguyên (đẩy ra xa nhau).
- `"RL"`, `"LR"` kề nhau → giữ nguyên.
- Đầu `.` rồi `L`; cuối `R` rồi `.`.
- Nhiều đoạn liên tiếp: `"R.LR.L"`.

In [19]:
cases = {
    ".": ".", "L": "L", "R": "R", "....": "....",
    "R.": "RR", ".L": "LL", "L.": "L.", ".R": ".R",
    "R.L": "R.L", "R..L": "RRLL", "R...L": "RR.LL", "R....L": "RRRLLL",
    "L.R": "L.R", "L..R": "L..R", "RL": "RL", "LR": "LR",
    "..L": "LLL", "R..": "RRR", "..L..": "LLL..", "..R..": "..RRR",
    "R.LR.L": "R.LR.L", "RR.L": "RR.L", "R.L.L": "R.LLL", "R.R.L": "RRR.L",
}
for s, want in cases.items():
    check(s, want)
    assert push_brute(s) == want, (s, push_brute(s), want)
print(len(cases), "edge case viết tay: cả", len(IMPLS), "cài đặt đúng")

24 edge case viết tay: cả 8 cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_inplace(dominoes):                              # mô phỏng nhưng cập nhật tại chỗ
    s = dominoes
    while True:
        t = step_inplace(s)
        if t == s:
            return s
        s = t


def bug_no_tie(dominoes):                               # R..L: ô giữa (số ô lẻ) bị gán R thay vì đứng yên
    s = "L" + dominoes + "R"
    ans, i = [], 0
    for j in range(1, len(s)):
        if s[j] == ".":
            continue
        gap = j - i - 1
        if i:
            ans.append(s[i])
        if s[i] == s[j]:
            ans.append(s[i] * gap)
        elif s[i] == "L" and s[j] == "R":
            ans.append("." * gap)
        else:
            ans.append("R" * ((gap + 1) // 2) + "L" * (gap // 2))     # SAI: ô giữa gán R
        i = j
    return "".join(ans)


def bug_no_sentinel(dominoes):                          # bỏ lính canh: không xử lý '.' ở hai đầu
    marks = [i for i, c in enumerate(dominoes) if c != "."]
    ans = list(dominoes)
    for a, b in zip(marks, marks[1:]):
        gap = b - a - 1
        sa, sb = dominoes[a], dominoes[b]
        if sa == sb:
            ans[a + 1:b] = [sa] * gap
        elif sa == "R" and sb == "L":
            ans[a + 1:a + 1 + gap // 2] = ["R"] * (gap // 2)
            ans[b - gap // 2:b] = ["L"] * (gap // 2)
    return "".join(ans)


def bug_L_to_R_pull(dominoes):                          # L..R: cho rằng chúng kéo nhau lại (đổ vào giữa)
    s = "L" + dominoes + "R"
    ans, i = [], 0
    for j in range(1, len(s)):
        if s[j] == ".":
            continue
        gap = j - i - 1
        if i:
            ans.append(s[i])
        if s[i] == s[j]:
            ans.append(s[i] * gap)
        elif s[i] == "L" and s[j] == "R":
            ans.append("L" * (gap // 2) + "R" * (gap - gap // 2))        # SAI: đổ vào giữa
        else:
            ans.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))
        i = j
    return "".join(ans)


def bug_force_no_reset(dominoes):                       # lực R không bị L chặn
    n = len(dominoes)
    fr, fl = [0] * n, [0] * n
    f = 0
    for i, c in enumerate(dominoes):
        f = n if c == "R" else max(f - 1, 0)            # SAI: bỏ nhánh L đặt về 0
        fr[i] = f
    f = 0
    for i in range(n - 1, -1, -1):
        c = dominoes[i]
        f = n if c == "L" else 0 if c == "R" else max(f - 1, 0)
        fl[i] = f
    return "".join("R" if fr[i] > fl[i] else "L" if fr[i] < fl[i] else "." for i in range(n))


def bug_bfs_overwrite(dominoes):                        # BFS không phát hiện va chạm: ghi đè
    n = len(dominoes)
    ans = list(dominoes)
    seen = [c != "." for c in dominoes]
    q = deque(i for i, c in enumerate(dominoes) if c != ".")
    while q:
        i = q.popleft()
        j = i - 1 if ans[i] == "L" else i + 1
        if 0 <= j < n and not seen[j]:
            seen[j] = True
            ans[j] = ans[i]
            q.append(j)
    return "".join(ans)


BUGS = [bug_inplace, bug_no_tie, bug_no_sentinel, bug_L_to_R_pull, bug_force_no_reset, bug_bfs_overwrite]
rng = random.Random(1)
tests = ["".join(rng.choices("LR.", weights=(1, 1, 3), k=rng.randint(1, 9))) for _ in range(800)]
for bug in BUGS:
    bad = next((t for t in tests if bug(t) != push_brute(t)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên {bad!r}: ra {bug(bad)!r}, đúng {push_brute(bad)!r}")

bug_inplace          sai trên 'R..LRL.L': ra 'RR.LRLLL', đúng 'RRLLRLLL'
bug_no_tie           sai trên 'R.LR': ra 'RRLR', đúng 'R.LR'
bug_no_sentinel      sai trên '..L': ra '..L', đúng 'LLL'
bug_L_to_R_pull      sai trên '..': ra 'LR', đúng '..'
bug_force_no_reset   sai trên 'RR.RL.': ra 'RRRRLR', đúng 'RRRRL.'
bug_bfs_overwrite    sai trên 'R.LR': ra 'RRLR', đúng 'R.LR'


### D. Vét cạn: mọi chuỗi `L`, `R`, `.` độ dài 1..9

Có `3 + 9 + 27 + … + 19683 = 29523` chuỗi. Mọi cài đặt phải khớp mô phỏng từng giây trên **tất cả**.

In [21]:
t0 = time.perf_counter()
cnt = 0
for L in range(1, 10):
    for t in product("LR.", repeat=L):
        s = "".join(t)
        want = push_brute(s)
        for name in FAST:
            assert IMPLS[name](s) == want, (name, s)
        cnt += 1
print(cnt, "chuỗi vét cạn: 7 cài đặt nhanh khớp oracle mô phỏng; mất", round(time.perf_counter() - t0, 1), "s")
assert cnt == sum(3 ** L for L in range(1, 10))

29523 chuỗi vét cạn: 7 cài đặt nhanh khớp oracle mô phỏng; mất 0.6 s


### E. Ngẫu nhiên có seed và tính chất

Chuỗi ngẫu nhiên với nhiều mật độ `.` khác nhau, độ dài tới `3000` (brute vẫn chạy được), rồi các **tính chất** (metamorphic):

1. Kết quả **ổn định**: áp dụng thêm một giây mô phỏng không đổi nữa.
2. Quân `L`/`R` ban đầu **giữ nguyên** vị trí và ký tự.
3. **Đối xứng gương**: đảo chuỗi rồi đổi `L ↔ R` cho kết quả là gương của kết quả cũ.
4. Số ký tự `.` không tăng.

In [22]:
rng = random.Random(20240838)
count = 0
for _ in range(1500):
    n = rng.randint(1, 3000) if rng.random() < 0.2 else rng.randint(1, 60)
    w = rng.choice([(1, 1, 8), (1, 1, 1), (1, 3, 6), (3, 1, 6), (1, 1, 30), (5, 5, 1)])
    s = random_dominoes(rng, n, w)
    want = push_brute(s)
    for name in FAST:
        assert IMPLS[name](s) == want, (name, s)
    count += 1
print(count, "chuỗi ngẫu nhiên (tới 3000): mọi cài đặt trùng mô phỏng")

mirror = lambda t: t[::-1].translate(str.maketrans("LR", "RL"))
for _ in range(500):
    s = random_dominoes(rng, rng.randint(1, 80), (1, 1, 4))
    r = Solution().pushDominoes(s)
    assert step_simultaneous(r) == r                                     # ổn định
    assert all(r[i] == c for i, c in enumerate(s) if c != ".")           # quân bị đẩy giữ nguyên
    assert Solution().pushDominoes(mirror(s)) == mirror(r)               # đối xứng gương
    assert r.count(".") <= s.count(".")
    assert Solution().pushDominoes(r) == r                               # idempotent
print("tính chất: ổn định, giữ quân bị đẩy, đối xứng gương, idempotent — OK trên 500 chuỗi")

1500 chuỗi ngẫu nhiên (tới 3000): mọi cài đặt trùng mô phỏng
tính chất: ổn định, giữ quân bị đẩy, đối xứng gương, idempotent — OK trên 500 chuỗi


### F. Giới hạn lớn nhất

`n = 10^5`: ngẫu nhiên, toàn `.`, `"R" + "."*(n−1)`, `"."*(n−1) + "L"`, `"R" + "."*(n−2) + "L"` (hai bên gặp nhau ở giữa), xen kẽ `"RL"`, `"LR"`.

In [23]:
n = 10 ** 5
cases = {
    "ngẫu nhiên": random_dominoes(random.Random(1), n),
    "toàn chấm": "." * n,
    "R rồi chấm": "R" + "." * (n - 1),
    "chấm rồi L": "." * (n - 1) + "L",
    "R chấm L (n chẵn)": "R" + "." * (n - 2) + "L",
    "R chấm L (n lẻ)": "R" + "." * (n - 3) + "L",
    "RL xen kẽ": "RL" * (n // 2),
    "LR xen kẽ": "LR" * (n // 2),
    "R.L lặp": "R.L" * (n // 3),
}
for name, s in cases.items():
    ref = push_segments(s)
    for impl in FAST:
        assert IMPLS[impl](s) == ref, (name, impl)
    assert len(ref) == len(s)
    print(f"{name:20s}: {ref[:12]!r}...{ref[-6:]!r}  (7 cài đặt khớp)")
assert push_segments("R" + "." * (n - 1)) == "R" * n
assert push_segments("." * n) == "." * n
mid = push_segments("R" + "." * (n - 2) + "L")
assert mid == "R" * (n // 2) + "L" * (n // 2)
mid = push_segments("R" + "." * (n - 3) + "L")          # n - 3 chấm (lẻ): đúng một ô giữa đứng yên
assert mid[len(mid) // 2] == "." and mid.count(".") == 1

ngẫu nhiên          : 'RRRR.LLLLLLL'...'RLLL..'  (7 cài đặt khớp)
toàn chấm           : '............'...'......'  (7 cài đặt khớp)


R rồi chấm          : 'RRRRRRRRRRRR'...'RRRRRR'  (7 cài đặt khớp)
chấm rồi L          : 'LLLLLLLLLLLL'...'LLLLLL'  (7 cài đặt khớp)


R chấm L (n chẵn)   : 'RRRRRRRRRRRR'...'LLLLLL'  (7 cài đặt khớp)
R chấm L (n lẻ)     : 'RRRRRRRRRRRR'...'LLLLLL'  (7 cài đặt khớp)


RL xen kẽ           : 'RLRLRLRLRLRL'...'RLRLRL'  (7 cài đặt khớp)


LR xen kẽ           : 'LRLRLRLRLRLR'...'LRLRLR'  (7 cài đặt khớp)
R.L lặp             : 'R.LR.LR.LR.L'...'R.LR.L'  (7 cài đặt khớp)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cập nhật tại chỗ thay vì đồng thời

Mô phỏng một giây bằng cách sửa chính mảng rồi đọc luôn giá trị vừa sửa: sóng đổ lan **nhanh hơn** một ô/giây, và quan trọng hơn là **va chạm bị phát hiện sai**.

In [24]:
for s in ("R...", "R..L", ".L.R...LR..L.."):
    print(f"{s:16s} tại chỗ: {step_inplace(s)!r:16} | đồng thời: {step_simultaneous(s)!r}")
bad = next(t for t in tests if bug_inplace(t) != push_brute(t))
print("ví dụ kết quả cuối sai:", bad, "->", bug_inplace(bad), "| đúng:", push_brute(bad))
assert step_inplace("R...") == "RRRR" and step_simultaneous("R...") == "RR.."

R...             tại chỗ: 'RRRR'           | đồng thời: 'RR..'
R..L             tại chỗ: 'RR.L'           | đồng thời: 'RRLL'
.L.R...LR..L..   tại chỗ: 'LL.RRR.LRR.L..' | đồng thời: 'LL.RR.LLRRLL..'
ví dụ kết quả cuối sai: R..LRL.L -> RR.LRLLL | đúng: RRLLRLLL


**Vì sao sai:** trong một giây mỗi quân chỉ đẩy quân kề; cập nhật tại chỗ cho phép quân vừa đổ đẩy tiếp ngay trong cùng giây. **Đúng:** tạo mảng mới từ mảng cũ.

### Lỗi 2: quên ô giữa đứng yên khi số ô lẻ (`R...L`)

Với `R…L` có `g` ô, nếu `g` lẻ thì ô chính giữa chịu lực **cùng lúc** từ hai phía và đứng yên.

In [25]:
for s in ("R.L", "R...L", "R.....L"):
    print(f"{s:10s}: bug_no_tie = {bug_no_tie(s)!r:10} | đúng = {push_segments(s)!r}")
assert bug_no_tie("R...L") == "RRRLL" and push_segments("R...L") == "RR.LL"

R.L       : bug_no_tie = 'RRL'      | đúng = 'R.L'
R...L     : bug_no_tie = 'RRRLL'    | đúng = 'RR.LL'
R.....L   : bug_no_tie = 'RRRRLLL'  | đúng = 'RRR.LLL'


**Vì sao sai:** ô giữa cách đều hai nguồn nên nhận lực cùng lúc. **Đúng:** `"R" * (g//2) + "." * (g%2) + "L" * (g//2)`.

### Lỗi 3: quên các đoạn ở hai đầu (không có lính canh)

`"..L"`: hai quân `.` đầu hàng bị `L` kéo đổ trái. `"R.."`: hai quân `.` cuối bị `R` đẩy phải. Nếu chỉ duyệt các cặp ký tự bị đẩy liền kề thì các đoạn biên bị bỏ sót.

In [26]:
for s in ("..L", "R..", ".L.R...LR..L.."):
    print(f"{s:16s}: không lính canh = {bug_no_sentinel(s)!r:16} | đúng = {push_segments(s)!r}")
assert bug_no_sentinel("..L") == "..L" and push_segments("..L") == "LLL"

..L             : không lính canh = '..L'            | đúng = 'LLL'
R..             : không lính canh = 'R..'            | đúng = 'RRR'
.L.R...LR..L..  : không lính canh = '.L.RR.LLRRLL..' | đúng = 'LL.RR.LLRRLL..'


**Vì sao sai:** đoạn biên chỉ có một quân bị đẩy ở một phía. **Đúng:** thêm lính canh `L` ở đầu và `R` ở cuối, hoặc xử lý biên riêng.

### Lỗi 4: nhầm `L…R` (đẩy ra xa) thành "kéo vào nhau"

`L` đổ sang **trái** và `R` đổ sang **phải**; ở giữa chúng không có lực nào, các quân `.` đứng yên.

In [27]:
for s in ("L..R", "L...R"):
    print(f"{s:8s}: nhầm = {bug_L_to_R_pull(s)!r:8} | đúng = {push_segments(s)!r}")
assert bug_L_to_R_pull("L..R") == "LLRR" and push_segments("L..R") == "L..R"

L..R    : nhầm = 'LLRR'   | đúng = 'L..R'
L...R   : nhầm = 'LLRRR'  | đúng = 'L...R'


**Vì sao sai:** hướng đổ quyết định ai bị đẩy: `L` chỉ đẩy quân bên **trái** nó, `R` chỉ đẩy quân bên **phải** nó. **Đúng:** đoạn giữa `L…R` giữ `.`.

### Lỗi 5: lực `R` lan qua một quân `L`

Khi tính lực bằng hai lượt quét, gặp `L` phải **đặt lại lực `R` về 0**: quân `L` đổ trái, chặn sóng `R`.

In [28]:
for s in ("R.L...", "RL..R.L"):
    print(f"{s:10s}: không reset = {bug_force_no_reset(s)!r:10} | đúng = {push_segments(s)!r}")
bad = next(t for t in tests if bug_force_no_reset(t) != push_brute(t))
print("ví dụ khác:", bad, "->", bug_force_no_reset(bad), "| đúng:", push_brute(bad))

R.L...    : không reset = 'R.LRRR'   | đúng = 'R.L...'
RL..R.L   : không reset = 'RLRRR.L'  | đúng = 'RL..R.L'
ví dụ khác: RR.RL. -> RRRRLR | đúng: RRRRL.


**Vì sao sai:** nếu lực `R` không bị chặn bởi `L`, nó "xuyên" qua `L` và đẩy các quân phía sau (sai). **Đúng:** `elif c == 'L': f = 0`.

### Lỗi 6: BFS không phát hiện va chạm cùng giây

Chỉ đánh dấu `seen` rồi ghi đè hướng thì quân bị đẩy cùng lúc từ hai phía lấy hướng của nguồn nào đến sau — không đứng yên. Phải ghi **thời điểm** và nhận ra lực thứ hai cùng thời điểm.

In [29]:
for s in ("R.L", "R...L"):
    print(f"{s:8s}: BFS ghi đè = {bug_bfs_overwrite(s)!r:8} | đúng = {push_segments(s)!r}")
assert bug_bfs_overwrite("R.L") != "R.L" and push_bfs("R.L") == "R.L"

R.L     : BFS ghi đè = 'RRL'    | đúng = 'R.L'
R...L   : BFS ghi đè = 'RRRLL'  | đúng = 'RR.LL'


**Vì sao sai:** thiếu thông tin thời điểm nên không phân biệt "tới trước" với "tới cùng lúc". **Đúng:** `time_at[j] == time_at[i] + 1` ⇒ cân bằng `'.'`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "nhiều nguồn lan tỏa đồng thời theo hướng, gặp nhau thì triệt tiêu" → **chia theo các nguồn liền kề** (hai con trỏ), hoặc **mảng lực/thời điểm** (hai lượt quét), hoặc **BFS nhiều nguồn**.

**Mẹo:**

1. Hỏi: ảnh hưởng của một nguồn bị **chặn** ở đâu? Thường chỉ cần hai nguồn gần nhất hai bên.
2. Dùng **lính canh** để gộp trường hợp biên vào trường hợp tổng quát.
3. Mô phỏng nên cập nhật **đồng thời** (mảng mới từ mảng cũ).
4. Với "ai tới trước thắng, cùng lúc hòa": so sánh khoảng cách / thời điểm.
5. Kiểm tra bằng **vét cạn mọi chuỗi ngắn** (`3^9 ≈ 2·10^4`) và các tính chất (đối xứng gương, idempotent, ổn định).
6. `"".join` danh sách đoạn nhanh hơn ghép chuỗi bằng `+=` trong vòng lặp lớn.

**Họ bài cùng pattern:** Rotting Oranges (994, BFS nhiều nguồn), Walls and Gates (286), Trapping Rain Water (42, hai lượt quét), Product of Array Except Self (238), Candy (135).

## 15. Biến thể

1. **Thời gian ổn định:** sau bao nhiêu giây thì hết thay đổi (bài tập 5).
2. **Trạng thái sau đúng `t` giây** (không phải trạng thái cuối).
3. **Đếm quân còn đứng** ở trạng thái cuối.
4. **Ba loại lực** hoặc lực giảm dần theo khoảng cách (như mảng "lực" ở Cách 3, mở rộng).
5. **Hàng domino vòng tròn** (ô cuối nối ô đầu): cần xử lý vòng.

Cài đặt biến thể 2 và 3 và đối chiếu mô phỏng:

In [30]:
def state_after(dominoes, t):
    """Biến thể 2: trạng thái sau đúng t giây (mô phỏng, hoặc trạng thái cuối nếu đã ổn định)."""
    s = dominoes
    for _ in range(t):
        nxt = step_simultaneous(s)
        if nxt == s:
            break
        s = nxt
    return s


def standing(dominoes):
    """Biến thể 3: số quân còn đứng ở trạng thái cuối."""
    return Solution().pushDominoes(dominoes).count(".")


s = ".L.R...LR..L.."
for t in range(0, 4):
    print(f"sau {t} giây:", state_after(s, t))
assert state_after(s, 100) == Solution().pushDominoes(s) == ex2[1]
print("số quân còn đứng:", standing(s), "| đếm tay '.' trong", ex2[1], "=", ex2[1].count("."))
assert standing(s) == 4 and standing("R...L") == 1

sau 0 giây: .L.R...LR..L..
sau 1 giây: LL.RR.LLRRLL..
sau 2 giây: LL.RR.LLRRLL..
sau 3 giây: LL.RR.LLRRLL..
số quân còn đứng: 4 | đếm tay '.' trong LL.RR.LLRRLL.. = 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng giây | Mảng mới từ mảng cũ mỗi giây | `O(n²)` | `O(n)` | Oracle; `n` nhỏ |
| 2. BFS nhiều nguồn | Thời điểm đổ + phát hiện va chạm | `O(n)` | `O(n)` | Khi cần thời điểm đổ của từng quân |
| 3. Hai lượt quét lực | Lực `R` giảm dần phải, lực `L` giảm dần trái | `O(n)` | `O(n)` | Ngắn, dễ nhớ |
| 4. Chia đoạn + lính canh | Xử lý từng đoạn `.` giữa hai quân bị đẩy | `O(n)` | `O(n)` | **Đáp án chính**; chứng minh rõ |

Cả bốn khớp nhau trên 29 523 chuỗi vét cạn và hàng nghìn chuỗi ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
for s in ("R.L", "R..L", ".L.R.", "RR.LL", "L.R.L"):
    print(f"{s:6s} -> {Solution().pushDominoes(s)}")

R.L    -> R.L
R..L   -> RRLL
.L.R.  -> LL.RR
RR.LL  -> RR.LL
L.R.L  -> L.R.L


### Bài 2 (tính tay)

- (a) `"R....L.R..."`: chia thành các đoạn và tính kết quả.
- (b) Có bao nhiêu quân còn đứng ở kết quả?
- (c) Sau **1 giây** trạng thái là gì (mô phỏng tay)?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai lượt quét lực:

In [32]:
def force_todo(dominoes):
    n = len(dominoes)
    right, left = [0] * n, [0] * n
    f = 0
    for i, c in enumerate(dominoes):
        pass            # TODO 1: R -> f = n; L -> f = 0; '.' -> f = max(f - 1, 0); lưu right[i] = f
    f = 0
    for i in range(n - 1, -1, -1):
        pass            # TODO 2: L -> f = n; R -> f = 0; '.' -> f = max(f - 1, 0); lưu left[i] = f
    return "".join("R" if right[i] > left[i] else "L" if right[i] < left[i] else "." for i in range(n))


# Khi làm xong, bỏ comment để kiểm tra:
# assert force_todo(".L.R...LR..L..") == "LL.RR.LLRRLL.." and force_todo("RR.L") == "RR.L"

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng chia đoạn (hai con trỏ) hoặc BFS:

In [33]:
class MySolution:
    def pushDominoes(self, dominoes):
        pass            # TODO


# Khi làm xong:
# assert MySolution().pushDominoes("RR.L") == "RR.L"
# assert MySolution().pushDominoes(".L.R...LR..L..") == "LL.RR.LLRRLL.."

### Bài 5 (mini challenge): thời gian ổn định

Viết `seconds_until_stable(dominoes)` trả về số **giây** cho tới khi không còn thay đổi (số lần mô phỏng thật sự làm thay đổi trạng thái), trong `O(n)`, **không mô phỏng**. Gợi ý: chia đoạn như Cách 4; thời gian của mỗi đoạn là: `R…L` với `g` ô: `g // 2`; `R…R` hoặc `L…L` với `g` ô: `g`; `L…R`: `0`; đoạn biên (từ đầu tới `L` đầu tiên hoặc từ `R` cuối tới hết): số ô `.` nếu bị kéo/đẩy, ngược lại `0`. Đáp án là **lớn nhất** các thời gian đoạn.

In [34]:
def seconds_until_stable(dominoes):
    pass            # TODO


# Khi làm xong:
# assert seconds_until_stable("R...L") == 1 and seconds_until_stable("R.....") == 5 and seconds_until_stable("RL") == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: xem bảng 4 trường hợp ở mục 4.
- Mức 2: `R.L` có đúng một ô giữa → đứng yên.
- Mức 3: `"R.L"` → `R.L` (một ô giữa); `"R..L"` → `RRLL`; `".L.R."` → `LL.R.`? hãy tính từng đoạn: đoạn đầu `.` + `L` đổ trái → `LL`; đoạn `L.R` đứng yên `.`; đoạn sau `R` + `.` đổ phải → `RR`: được `LL.RR`; `"RR.LL"` giữ nguyên; `"L.R.L"`: đoạn `L.R` đứng yên, đoạn `R.L` có một ô giữa đứng yên → giữ nguyên.

**Bài 2**

- Mức 1: các quân bị đẩy: `R@0`, `L@5`, `R@7`; đoạn giữa: `R....L` (4 ô), `L.R` (1 ô), sau `R@7` còn 3 ô.
- Mức 2: đoạn `R....L` có `g = 4` ô: `g // 2 = 2` ô đổ `R` rồi `2` ô đổ `L`, cộng hai quân `R`, `L` ở hai đầu.
- Mức 3: dùng bảng; kiểm bằng máy ở mục 19.

**Bài 3**

- Mức 1: giống mục 7 Cách 3.
- Mức 2: `f = n if c == 'R' else 0 if c == 'L' else max(f - 1, 0)`.
- Mức 3: lưu `right[i] = f` ngay sau khi tính.

**Bài 4**

- Mức 1: thêm lính canh `"L" + s + "R"`.
- Mức 2: với mỗi cặp liền kề `(i, j)` tính `gap = j - i - 1` và 4 trường hợp.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: nghĩ về từng đoạn độc lập: các đoạn đổ **đồng thời**, nên thời gian tổng là lớn nhất.
- Mức 2: `R…L` với `g` ô tốn `g // 2` giây (ô giữa lẻ không đổ).
- Mức 3: đoạn `L…L` hoặc `R…R` với `g` ô tốn `g` giây; kiểm bằng mô phỏng.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
expected = {"R.L": "R.L", "R..L": "RRLL", ".L.R.": "LL.RR", "RR.LL": "RR.LL", "L.R.L": "L.R.L"}
for s, want in expected.items():
    assert Solution().pushDominoes(s) == want == push_brute(s), (s, Solution().pushDominoes(s), want)
print("Bài 1 OK:", expected)

Bài 1 OK: {'R.L': 'R.L', 'R..L': 'RRLL', '.L.R.': 'LL.RR', 'RR.LL': 'RR.LL', 'L.R.L': 'L.R.L'}


### Bài 2

(a) `"R....L.R..."`: đoạn `R….L` (4 ô) → `R` `RR` `LL` `L` = `RRRLLL`; đoạn `L.R`: một ô đứng yên `.`; đoạn sau `R@7`: 3 ô đổ phải → `RRRR`. Kết quả `"RRRLLL.RRRR"`. (b) còn đúng 1 quân đứng. (c) sau 1 giây: `"RR..LL.RR.."`.

In [36]:
s = "R....L.R..."
res = Solution().pushDominoes(s)
print(res, "| còn đứng:", res.count("."), "| sau 1 giây:", state_after(s, 1))
assert res == push_brute(s) == "RRRLLL.RRRR"
assert res.count(".") == 1
assert state_after(s, 1) == "RR..LL.RR.."
print("Bài 2 OK")

RRRLLL.RRRR | còn đứng: 1 | sau 1 giây: RR..LL.RR..
Bài 2 OK


### Bài 3

In [37]:
def force_answer(dominoes):
    n = len(dominoes)
    right, left = [0] * n, [0] * n
    f = 0
    for i, c in enumerate(dominoes):
        f = n if c == "R" else 0 if c == "L" else max(f - 1, 0)
        right[i] = f
    f = 0
    for i in range(n - 1, -1, -1):
        c = dominoes[i]
        f = n if c == "L" else 0 if c == "R" else max(f - 1, 0)
        left[i] = f
    return "".join("R" if right[i] > left[i] else "L" if right[i] < left[i] else "." for i in range(n))


assert force_answer(".L.R...LR..L..") == "LL.RR.LLRRLL.." and force_answer("RR.L") == "RR.L"
rng = random.Random(3)
for _ in range(300):
    s = random_dominoes(rng, rng.randint(1, 40), (1, 1, 4))
    assert force_answer(s) == push_brute(s)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def pushDominoes(self, dominoes):
        s = "L" + dominoes + "R"
        out = []
        i = 0
        for j in range(1, len(s)):
            if s[j] == ".":
                continue
            gap = j - i - 1
            if i:
                out.append(s[i])
            if s[i] == s[j]:
                out.append(s[i] * gap)
            elif s[i] == "L":                       # L..R
                out.append("." * gap)
            else:                                   # R..L
                out.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))
            i = j
        return "".join(out)


assert MySolutionAnswer().pushDominoes("RR.L") == "RR.L"
assert MySolutionAnswer().pushDominoes(".L.R...LR..L..") == "LL.RR.LLRRLL.."
rng = random.Random(4)
for _ in range(300):
    s = random_dominoes(rng, rng.randint(1, 40), (1, 1, 4))
    assert MySolutionAnswer().pushDominoes(s) == push_brute(s)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Chia đoạn (có lính canh): thời gian đoạn `R…L` với `g` ô là `g // 2`; `L…L` hoặc `R…R`: `g`; `L…R`: `0`. Với lính canh đầu `L` và quân đầu tiên thật là `L`: đoạn `L…L` có `g` ô (số `.` đầu hàng) tốn `g` giây; nếu quân đầu là `R` thì `L…R` tốn `0`. Tương tự cuối hàng. Đáp án là lớn nhất các thời gian.

In [39]:
def seconds_until_stable_answer(dominoes):
    s = "L" + dominoes + "R"
    best, i = 0, 0
    for j in range(1, len(s)):
        if s[j] == ".":
            continue
        gap = j - i - 1
        if s[i] == s[j]:
            t = gap
        elif s[i] == "L":
            t = 0
        else:
            t = gap // 2
        best = max(best, t)
        i = j
    return best


assert seconds_until_stable_answer("R...L") == 1 and seconds_until_stable_answer("R.....") == 5 and seconds_until_stable_answer("RL") == 0
rng = random.Random(5)
for _ in range(2000):
    s = random_dominoes(rng, rng.randint(1, 40), (1, 1, 4))
    push_brute(s)
    assert seconds_until_stable_answer(s) == push_brute.seconds, (s, seconds_until_stable_answer(s), push_brute.seconds)
for L in range(1, 8):
    for t in product("LR.", repeat=L):
        s = "".join(t)
        push_brute(s)
        assert seconds_until_stable_answer(s) == push_brute.seconds, s
print("Bài 5 OK: khớp mô phỏng trên 2000 chuỗi ngẫu nhiên và mọi chuỗi độ dài <= 7")

Bài 5 OK: khớp mô phỏng trên 2000 chuỗi ngẫu nhiên và mọi chuỗi độ dài <= 7


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chia đoạn với lính canh):**

```python
class Solution:
    def pushDominoes(self, dominoes: str) -> str:
        s = "L" + dominoes + "R"
        ans, i = [], 0
        for j in range(1, len(s)):
            if s[j] == ".":
                continue
            gap = j - i - 1
            if i:
                ans.append(s[i])
            if s[i] == s[j]:
                ans.append(s[i] * gap)                                       # L..L / R..R
            elif s[i] == "L":
                ans.append("." * gap)                                        # L..R
            else:
                ans.append("R" * (gap // 2) + "." * (gap % 2) + "L" * (gap // 2))   # R..L
            i = j
        return "".join(ans)
```

**Checklist trước khi submit:**

- [ ] Xử lý **cả 4 cặp** `(R,L)`, `(R,R)`, `(L,L)`, `(L,R)` giữa hai quân bị đẩy?
- [ ] `R…L` có `g` lẻ: **ô giữa đứng yên** (`"." * (g % 2)`)?
- [ ] Có xử lý hai đầu (lính canh `L` ở đầu, `R` ở cuối) và **không** đưa lính canh vào kết quả?
- [ ] Nếu mô phỏng: cập nhật **đồng thời** (mảng mới từ mảng cũ)?
- [ ] Nếu quét lực: `L` đặt lại lực `R` về `0` (và ngược lại)?
- [ ] Đã thử `"RR.L"`, `"R.L"`, `"R...L"`, `".L.R...LR..L.."`, `"."`, `"..L"`?
- [ ] Đúng tên `class Solution`, `pushDominoes(self, dominoes)`, không in debug.